# COMP 472 Project: Feed-Forward Neural Network (PyTorch)

**Tasks**
- **Image**: 500-d features, 50 classes (16,000 train / 4,000 validation)
- **Sentiment**: 768-d BERT features, 3 classes (9,600 train / 2,400 validation)

**Validation**: a stratified 80/20 split of each training set (fixed seed 42). Feature scaling is fitted on the 80% training split only. The test set is never used to tune or select models, so the final score is a fair measure of performance on unseen data.

**Noise level**: with 4,000 image validation examples, differences under about 1.5 points can be chance. With 2,400 sentiment examples, under about 2 points.

**Storyline**
1. Floor: the untrained network
2. Baseline: the spec network, Input → 256 → 64 → Output
3. Experiment 1: is it overfitting? → dropout
4. Experiment 2: is the learning rate too high? → learning-rate sweep
5. Experiment 3: does network size matter? → size sweep
6. Improving image: feature scaling, label smoothing
7. Averaging several networks (both tasks)
8. Closing the gap on image: feature inspection, square-root scaling, input dropout, mixup, weight decay, and a 10-network check of near-misses
9. Conclusions

The "Result" notes quote the numbers from our run. A rerun on a GPU may differ slightly.

# Part A: Setup

## A1. Mount Drive and unzip the dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, zipfile

ZIP_PATH = '/content/drive/MyDrive/comp472/comp-472-project-fall-2026-concordia.zip'
DATA_DIR = '/content/data'

os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DATA_DIR)

def find_csv(name):
    """Find a CSV anywhere inside the extracted zip (handles nested folders)."""
    hits = [p for p in glob.glob(f'{DATA_DIR}/**/{name}', recursive=True) if '__MACOSX' not in p]
    if not hits:
        raise FileNotFoundError(f'{name} not found under {DATA_DIR}')
    return hits[0]

Mounted at /content/drive


## A2. Imports and reproducibility

In [2]:
import copy, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

Device: cuda


## A3. Data: load, stratified split, feature scaling

Scaling options: `'raw'` (none), `'standardize'` (each feature gets mean 0 and std 1, the default), `'l2'` (each example rescaled to the same length, keeping only its direction, which is what cosine distance in k-NN measures), and `'l2+standardize'`. Part E adds square-root variants for the image features: `'sqrt'`, `'sqrt+l2'`, `'sqrt+standardize'`.

In [3]:
def load_csv(name, has_label=True):
    df = pd.read_csv(find_csv(name))
    feat_cols = [c for c in df.columns if c.startswith('feature_')]
    X = df[feat_cols].to_numpy(dtype=np.float32)
    y = df['label'].to_numpy(dtype=np.int64) if has_label else None
    return df['id'].to_numpy(), X, y

_raw_cache = {}
def load_task(task):
    """Read a task's CSVs once and keep them in memory."""
    if task not in _raw_cache:
        _, X, y = load_csv(f'{task}_train.csv')
        test_ids, X_test, _ = load_csv(f'{task}_test.csv', has_label=False)
        _raw_cache[task] = (X, y, X_test, test_ids)
    return _raw_cache[task]

def stratified_split(y, val_frac=0.2, seed=42):
    """Hold out val_frac of every class so both splits keep the class balance."""
    rng = np.random.default_rng(seed)
    tr_idx, va_idx = [], []
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        n_val = int(round(len(idx) * val_frac))
        va_idx.append(idx[:n_val])
        tr_idx.append(idx[n_val:])
    return np.concatenate(tr_idx), np.concatenate(va_idx)

SCALINGS = ['raw', 'standardize', 'l2', 'l2+standardize']        # compared in C1
SQRT_SCALINGS = ['sqrt', 'sqrt+l2', 'sqrt+standardize']            # compared in E2

def l2_normalize(A):
    """Rescale every row to length sqrt(d): keeps its direction, values stay around +-1."""
    return A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-8) * np.sqrt(A.shape[1])

def prepare(task, scaling='standardize', val_frac=0.2, seed=42, verbose=True):
    if scaling not in SCALINGS + SQRT_SCALINGS:
        raise ValueError(f'scaling must be one of {SCALINGS + SQRT_SCALINGS}')
    X, y, X_test, test_ids = load_task(task)
    tr, va = stratified_split(y, val_frac, seed)
    X_tr, y_tr, X_va, y_va, X_te = X[tr], y[tr], X[va], y[va], X_test

    if scaling.startswith('sqrt'):                    # per-value: compresses large values
        if X.min() < 0:
            raise ValueError('sqrt scaling needs non-negative features')
        X_tr, X_va, X_te = np.sqrt(X_tr), np.sqrt(X_va), np.sqrt(X_te)
    if 'l2' in scaling:                               # per-row: needs no training statistics
        X_tr, X_va, X_te = l2_normalize(X_tr), l2_normalize(X_va), l2_normalize(X_te)
    if 'standardize' in scaling:                      # per-column: statistics from TRAIN split only
        mean, std = X_tr.mean(axis=0), X_tr.std(axis=0) + 1e-8
        X_tr, X_va, X_te = (X_tr - mean) / std, (X_va - mean) / std, (X_te - mean) / std

    f32 = lambda A: np.ascontiguousarray(A, dtype=np.float32)
    d = dict(X_tr=f32(X_tr), y_tr=y_tr, X_va=f32(X_va), y_va=y_va,
             X_test=f32(X_te), test_ids=test_ids, n_classes=int(y.max()) + 1, scaling=scaling)
    if verbose:
        print(f'{task:9s} | {scaling:14s} | train {d["X_tr"].shape} | val {d["X_va"].shape} | '
              f'test {d["X_test"].shape} | classes {d["n_classes"]}')
    return d

In [4]:
data = {task: prepare(task, 'standardize') for task in ['image', 'sentiment']}

image     | standardize    | train (16000, 500) | val (4000, 500) | test (5000, 500) | classes 50
sentiment | standardize    | train (9600, 768) | val (2400, 768) | test (5000, 768) | classes 3


## A4. Model: feed-forward network

In [5]:
class MLP(nn.Module):
    """[Input dropout] -> [Linear -> ReLU -> Dropout] x len(hidden) -> Linear (one score per class)."""
    def __init__(self, in_dim, hidden, n_classes, dropout=0.0, input_dropout=0.0):
        super().__init__()
        layers = [nn.Dropout(input_dropout)] if input_dropout > 0 else []   # blanks input features (Part E)
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))  # softmax is inside CrossEntropyLoss
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

## A5. Training loop

Mini-batch Adam, cross-entropy loss, early stopping on validation accuracy (patience 10). The weights from the best validation epoch are restored at the end.

Options, all off by default:
- `label_smoothing`: softens the training targets. At 0.1, the true class is trained toward about 90% instead of 100%.
- `input_dropout`: randomly blanks input features during training (Part E).
- `mixup_alpha`: trains on blends of two examples and their labels (Part E).
- `optimizer='adamw'` and `cosine=True`: weight decay applied directly to the weights, and a learning rate that gradually falls to 0 over the run (Part E).

In [6]:
@torch.no_grad()
def predict_proba(model, X, batch_size=2048):
    """Class probabilities, shape (N, n_classes)."""
    model.eval()                                   # dropout off
    out = []
    for i in range(0, len(X), batch_size):
        xb = torch.from_numpy(X[i:i + batch_size]).to(device)
        out.append(torch.softmax(model(xb), dim=1).cpu().numpy())
    return np.concatenate(out)

def predict(model, X):
    return predict_proba(model, X).argmax(axis=1)

def accuracy(model, X, y):
    return float((predict(model, X) == y).mean())

def train_model(d, cfg, verbose=True):
    set_seed(cfg['seed'])
    model = MLP(d['X_tr'].shape[1], cfg['hidden'], d['n_classes'], cfg['dropout'],
                cfg.get('input_dropout', 0.0)).to(device)
    Opt = torch.optim.AdamW if cfg.get('optimizer', 'adam') == 'adamw' else torch.optim.Adam
    optimizer = Opt(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    scheduler = (torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'])
                 if cfg.get('cosine', False) else None)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=cfg.get('label_smoothing', 0.0))
    alpha = cfg.get('mixup_alpha', 0.0)
    loader = DataLoader(TensorDataset(torch.from_numpy(d['X_tr']), torch.from_numpy(d['y_tr'])),
                        batch_size=cfg['batch_size'], shuffle=True)

    best = dict(val_acc=-1.0, epoch=0, train_acc=None, state=None)
    history, epochs_without_improvement = [], 0

    for epoch in range(1, cfg['epochs'] + 1):
        model.train()                              # dropout on
        running_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            if alpha > 0:                          # mixup: blend each example with a random partner
                lam = float(np.random.beta(alpha, alpha))
                perm = torch.randperm(len(xb), device=device)
                out = model(lam * xb + (1 - lam) * xb[perm])
                loss = lam * loss_fn(out, yb) + (1 - lam) * loss_fn(out, yb[perm])
            else:
                loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(xb)
        if scheduler is not None:
            scheduler.step()

        train_loss = running_loss / len(d['X_tr'])
        train_acc = accuracy(model, d['X_tr'], d['y_tr'])
        val_acc = accuracy(model, d['X_va'], d['y_va'])
        history.append(dict(epoch=epoch, train_loss=train_loss, train_acc=train_acc, val_acc=val_acc))

        if val_acc > best['val_acc']:
            best.update(val_acc=val_acc, epoch=epoch, train_acc=train_acc,
                        state=copy.deepcopy(model.state_dict()))
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if verbose and (epoch == 1 or epoch % 5 == 0):
            print(f'  epoch {epoch:3d} | loss {train_loss:.4f} | train acc {train_acc:.4f} | val acc {val_acc:.4f}')
        if epochs_without_improvement >= cfg['patience']:
            if verbose:
                print(f'  early stop at epoch {epoch} (best epoch {best["epoch"]})')
            break

    model.load_state_dict(best['state'])           # keep the best-on-validation weights
    return model, best, pd.DataFrame(history)

## A6. Baseline configuration and experiment helpers

- `sweep(param, values)`: trains one network per value, changing only that setting. `gap` = train accuracy − validation accuracy; a large gap means memorization.
- `compare_scalings(task, cfg)`: one network per feature scaling.
- `seed_ensemble(d, cfg, seeds)`: trains one network per seed and averages their predicted probabilities, reporting accuracy when averaging the first 1, 2, … networks.
- `pick(df, param, off_value)`: the adoption rule used in Part E. Take the best value only if it beats the "off" setting by more than 1.5 points.

In [7]:
BASE_CFG = dict(
    hidden=[256, 64],     # architecture from the project spec
    dropout=0.0,
    lr=1e-3,
    weight_decay=0.0,
    label_smoothing=0.0,
    input_dropout=0.0,    # Part E
    mixup_alpha=0.0,      # Part E
    optimizer='adam',     # 'adamw' in Part E
    cosine=False,         # Part E
    batch_size=128,
    epochs=100,           # upper bound
    patience=10,
    seed=42,
)
NOISE_MARGIN = 0.015      # image differences smaller than this can be chance

def result_row(best):
    return dict(best_epoch=best['epoch'],
                train_acc=round(best['train_acc'], 4),
                val_acc=round(best['val_acc'], 4),
                gap=round(best['train_acc'] - best['val_acc'], 4))

def sweep(param, values, tasks=('image', 'sentiment'), base=None):
    base = BASE_CFG if base is None else base
    rows = []
    for task in tasks:
        for v in values:
            print(f'training: {task} | {param} = {v}')
            _, best, _ = train_model(data[task], {**base, param: v}, verbose=False)
            rows.append({'task': task, param: str(v) if isinstance(v, list) else v, **result_row(best)})
    return pd.DataFrame(rows)

def compare_scalings(task, cfg, scalings):
    rows = []
    for s in scalings:
        print(f'training: {task} | scaling = {s}')
        _, best, _ = train_model(prepare(task, s, verbose=False), cfg, verbose=False)
        rows.append(dict(scaling=s, **result_row(best)))
    return pd.DataFrame(rows)

def seed_ensemble(d, cfg, seeds):
    models, probs, rows = [], [], []
    for s in seeds:
        print(f'training: seed {s}')
        model, best, _ = train_model(d, {**cfg, 'seed': s}, verbose=False)
        models.append(model)
        probs.append(predict_proba(model, d['X_va']))
        rows.append(dict(seed=s, best_epoch=best['epoch'], val_acc=round(best['val_acc'], 4)))
    curve = pd.DataFrame([dict(n_networks=k,
                               val_acc=round(float((np.mean(probs[:k], axis=0).argmax(axis=1) == d['y_va']).mean()), 4))
                          for k in range(1, len(probs) + 1)])
    return models, pd.DataFrame(rows), curve

def pick(df, param, off_value, reference=None):
    """Adopt the best value only if it beats the reference by more than NOISE_MARGIN.
    The reference defaults to the row where the technique is off."""
    if reference is None:
        reference = lookup(df, **{param: off_value})
    best = df.loc[df['val_acc'].idxmax()]
    if best['val_acc'] - reference > NOISE_MARGIN:
        return float(best[param]), True
    return off_value, False

def lookup(df, value_col='val_acc', **match):
    """Pick one value out of a results table, e.g. lookup(arch, task='image', hidden='[]')."""
    mask = np.ones(len(df), dtype=bool)
    for col, val in match.items():
        mask &= (df[col] == val).to_numpy()
    return float(df.loc[mask, value_col].iloc[0])

# Part B: Experiments

## B1. Floor: the untrained network

**Question.** What does the network score before it learns anything?

**Experiment.** The baseline network with random weights and no training, averaged over 5 random initializations.

In [8]:
untrained = {}
for task in ['image', 'sentiment']:
    d, accs = data[task], []
    for s in range(5):
        set_seed(s)
        model = MLP(d['X_tr'].shape[1], BASE_CFG['hidden'], d['n_classes']).to(device)   # no training
        accs.append(accuracy(model, d['X_va'], d['y_va']))
    untrained[task] = float(np.mean(accs))
    print(f'{task:9s} | chance {1 / d["n_classes"]:.4f} | untrained {untrained[task]:.4f}')

image     | chance 0.0200 | untrained 0.0210
sentiment | chance 0.3333 | untrained 0.3326


**Result.** Image 2.1% (chance 2.0%). Sentiment 33.3% (chance 33.3%).

**Takeaway.** Untrained, the network is just guessing. This is the floor every later result is measured from.

## B2. Baseline: Input → 256 → 64 → Output

**Question.** How well does the spec architecture do with default settings?

**Experiment.** Adam, learning rate 1e-3, batch size 128, no dropout, early stopping.

In [9]:
baseline = {}
for task in ['image', 'sentiment']:
    print(f'=== {task} ===')
    _, best, history = train_model(data[task], BASE_CFG)
    baseline[task] = dict(best=best, history=history)

baseline_table = pd.DataFrame([dict(task=t, **result_row(b['best'])) for t, b in baseline.items()])
print(baseline_table.to_string(index=False))
print(f'\nOverall (image + sentiment) / 2 = {baseline_table["val_acc"].mean():.4f}')

=== image ===
  epoch   1 | loss 3.5913 | train acc 0.2824 | val acc 0.2132
  epoch   5 | loss 1.3673 | train acc 0.7709 | val acc 0.3252
  epoch  10 | loss 0.2660 | train acc 0.9911 | val acc 0.3350
  epoch  15 | loss 0.0334 | train acc 1.0000 | val acc 0.3355
  epoch  20 | loss 0.0122 | train acc 1.0000 | val acc 0.3342
  early stop at epoch 24 (best epoch 14)
=== sentiment ===
  epoch   1 | loss 0.8023 | train acc 0.7177 | val acc 0.6700
  epoch   5 | loss 0.4187 | train acc 0.8856 | val acc 0.6521
  epoch  10 | loss 0.0757 | train acc 0.9894 | val acc 0.6217
  early stop at epoch 11 (best epoch 1)
     task  best_epoch  train_acc  val_acc    gap
    image          14     1.0000    0.338 0.6620
sentiment           1     0.7177    0.670 0.0477

Overall (image + sentiment) / 2 = 0.5040


**Result.**

| Task | Best epoch | Train acc | Val acc |
|---|---|---|---|
| Image | 14 | 100% | 33.8% |
| Sentiment | 1 | 71.8% | 67.0% |

Overall: 0.504.

**Takeaway.** Training lifts both tasks far above the floor. But the image network memorizes the training set (100% vs 33.8%), and sentiment peaks after a single epoch. Each experiment below tests one explanation.

## B3. Experiment 1: dropout

**Question.** If overfitting is what limits validation accuracy, reducing it should raise validation accuracy. Does it?

**Experiment.** Dropout probability 0, 0.2, 0.3, 0.5 on both hidden layers.

In [10]:
dropout_results = sweep('dropout', [0.0, 0.2, 0.3, 0.5])
print(dropout_results.to_string(index=False))

training: image | dropout = 0.0
training: image | dropout = 0.2
training: image | dropout = 0.3
training: image | dropout = 0.5
training: sentiment | dropout = 0.0
training: sentiment | dropout = 0.2
training: sentiment | dropout = 0.3
training: sentiment | dropout = 0.5
     task  dropout  best_epoch  train_acc  val_acc    gap
    image      0.0          14     1.0000   0.3380 0.6620
    image      0.2          15     0.9164   0.3345 0.5819
    image      0.3          17     0.8447   0.3215 0.5232
    image      0.5          36     0.7283   0.3020 0.4263
sentiment      0.0           1     0.7177   0.6700 0.0477
sentiment      0.2           1     0.7074   0.6704 0.0370
sentiment      0.3           1     0.7034   0.6696 0.0339
sentiment      0.5          13     0.8517   0.6775 0.1742


**Result.**
- Image: the gap shrank from 66 to 43 points, but validation accuracy didn't rise: 33.8% → 33.5% → 32.2% → 30.2%.
- Sentiment: 67.0% to 67.8% across all values, within noise. At 0.5 the best epoch moved from 1 to 13, but it ended up in the same place.

**Takeaway.** Dropout reduces memorization without improving accuracy on new data. Fighting overfitting this way doesn't help.

## B4. Experiment 2: learning rate

**Question.** Sentiment peaks after one epoch. Is the learning rate too high?

**Experiment.** Learning rates 1e-2, 1e-3, 3e-4, 1e-4.

In [11]:
lr_results = sweep('lr', [1e-2, 1e-3, 3e-4, 1e-4])
print(lr_results.to_string(index=False))

training: image | lr = 0.01
training: image | lr = 0.001
training: image | lr = 0.0003
training: image | lr = 0.0001
training: sentiment | lr = 0.01
training: sentiment | lr = 0.001
training: sentiment | lr = 0.0003
training: sentiment | lr = 0.0001
     task     lr  best_epoch  train_acc  val_acc    gap
    image 0.0100          10     0.5374   0.2117 0.3257
    image 0.0010          14     1.0000   0.3380 0.6620
    image 0.0003          15     0.8134   0.3327 0.4806
    image 0.0001          40     0.7575   0.3257 0.4317
sentiment 0.0100           1     0.6984   0.6650 0.0334
sentiment 0.0010           1     0.7177   0.6700 0.0477
sentiment 0.0003           2     0.7338   0.6704 0.0633
sentiment 0.0001           4     0.7211   0.6717 0.0495


**Result.**
- Image: 1e-2 is too high (21.2%). 1e-3, 3e-4 and 1e-4 give 33.8%, 33.3% and 32.6%, all within noise.
- Sentiment: 66.5% to 67.2% for every rate. Lower rates delay the peak (epoch 4 at 1e-4) but don't raise it.

**Takeaway.** The learning rate isn't what limits either task. Keep 1e-3, the fastest of the good values.

## B5. Experiment 3: network size

**Question.** Does the network's capacity matter? `hidden` lists the hidden-layer sizes; `[]` means no hidden layers (a linear model).

**Experiment.** `[]`, `[64]`, `[256, 64]`, `[512, 256]`.

In [12]:
arch_results = sweep('hidden', [[], [64], [256, 64], [512, 256]])
print(arch_results.to_string(index=False))

training: image | hidden = []
training: image | hidden = [64]
training: image | hidden = [256, 64]
training: image | hidden = [512, 256]
training: sentiment | hidden = []
training: sentiment | hidden = [64]
training: sentiment | hidden = [256, 64]
training: sentiment | hidden = [512, 256]
     task     hidden  best_epoch  train_acc  val_acc    gap
    image         []           7     0.5427   0.3125 0.2302
    image       [64]           7     0.5721   0.2985 0.2736
    image  [256, 64]          14     1.0000   0.3380 0.6620
    image [512, 256]          16     1.0000   0.3635 0.6365
sentiment         []           3     0.7036   0.6729 0.0307
sentiment       [64]           2     0.7390   0.6733 0.0656
sentiment  [256, 64]           1     0.7177   0.6700 0.0477
sentiment [512, 256]           3     0.8194   0.6696 0.1498


**Result.**

| `hidden` | Image val | Sentiment val |
|---|---|---|
| `[]` (linear) | 31.3% | 67.3% |
| `[64]` | 29.9% | 67.3% |
| `[256, 64]` | 33.8% | 67.0% |
| `[512, 256]` | **36.4%** | 67.0% |

**Takeaway.**
- **Sentiment:** a linear model scores the same as every network. Hidden layers add nothing, so the limit is in the BERT features, not the model.
- **Image:** squeezing everything through one narrow 64-unit layer hurts, even compared with the linear model. A wider network helps (+2.6 points). Image has room left, so the rest of the work focuses there, starting from `[512, 256]`.

# Part C: Improving the image task

## C1. Feature scaling

**Question.** Our teammates' k-NN beats this network on image, and k-NN depends entirely on distances between vectors. Does how we scale the features matter?

**Experiment.** The `[512, 256]` network with each of the four scalings.

In [13]:
WIDE = {**BASE_CFG, 'hidden': [512, 256]}
scaling_results = compare_scalings('image', WIDE, SCALINGS)
print(scaling_results.to_string(index=False))

training: image | scaling = raw
training: image | scaling = standardize
training: image | scaling = l2
training: image | scaling = l2+standardize
       scaling  best_epoch  train_acc  val_acc    gap
           raw          41        1.0   0.3450 0.6550
   standardize          16        1.0   0.3635 0.6365
            l2          15        1.0   0.3690 0.6310
l2+standardize          18        1.0   0.3670 0.6330


**Result.** L2 scored 36.9% vs 36.4% for standardize, within noise. Raw and L2+standardize: see the table.

**Takeaway.** Scaling has no clear effect. L2 is kept because it scored highest and matches the idea behind cosine distance.

In [14]:
SCALING = 'l2'            # chosen from the table above
data['image'] = prepare('image', SCALING)

image     | l2             | train (16000, 500) | val (4000, 500) | test (5000, 500) | classes 50


## C2. Label smoothing

**Question.** Does discouraging extreme confidence reduce memorization enough to help?

**Experiment.** Label smoothing 0, 0.1, 0.2 on the `[512, 256]` network with L2 scaling.

In [15]:
ls_results = sweep('label_smoothing', [0.0, 0.1, 0.2], tasks=('image',), base=WIDE)
print(ls_results.to_string(index=False))

training: image | label_smoothing = 0.0
training: image | label_smoothing = 0.1
training: image | label_smoothing = 0.2
 task  label_smoothing  best_epoch  train_acc  val_acc    gap
image              0.0          15        1.0   0.3690 0.6310
image              0.1          17        1.0   0.3725 0.6275
image              0.2          16        1.0   0.3713 0.6287


**Result.** 0.1 scored 37.3% vs 36.9% without smoothing, within noise.

**Takeaway.** No clear effect. 0.1 is kept as the best-scoring value.

In [16]:
LABEL_SMOOTHING = 0.1     # chosen from the table above
IMAGE_CFG = {**WIDE, 'label_smoothing': LABEL_SMOOTHING}

# Part D: Averaging several networks

## D1. Image: averaging up to 10 networks

**Question.** Each network memorizes the training set in its own random way, so each makes different mistakes. Does averaging several networks cancel those mistakes out?

**Experiment.** Train 10 networks that differ only in random seed (seeds 0–9). Average their predicted probabilities, and measure accuracy when using the first 1, 2, …, 10 of them. The 5-network row uses the same five networks as our earlier 5-network run.

In [17]:
image_models, image_singles, image_curve = seed_ensemble(data['image'], IMAGE_CFG, seeds=range(10))
print(image_singles.to_string(index=False))
print(f'\nSingle networks: mean {image_singles["val_acc"].mean():.4f} '
      f'(min {image_singles["val_acc"].min():.4f}, max {image_singles["val_acc"].max():.4f})\n')
print(image_curve.to_string(index=False))

training: seed 0
training: seed 1
training: seed 2
training: seed 3
training: seed 4
training: seed 5
training: seed 6
training: seed 7
training: seed 8
training: seed 9
 seed  best_epoch  val_acc
    0          15   0.3745
    1          22   0.3748
    2          15   0.3703
    3          11   0.3723
    4          17   0.3817
    5          17   0.3790
    6          17   0.3688
    7          13   0.3715
    8          13   0.3730
    9          25   0.3740

Single networks: mean 0.3740 (min 0.3688, max 0.3817)

 n_networks  val_acc
          1   0.3745
          2   0.4133
          3   0.4300
          4   0.4305
          5   0.4420
          6   0.4495
          7   0.4527
          8   0.4577
          9   0.4567
         10   0.4582


**Result.** Average of 5 networks: 44.2%. Average of 10: 45.8%, up from 37.4% for a single network on average. The gains slow down: about +7 points from 1 to 5 networks, then +1.6 from 5 to 10, so adding more networks would bring little.

**Takeaway.** Averaging is by far the biggest single gain on image, about +8.4 points with 10 networks. The networks' instability was a large part of the error, and averaging fixes it where dropout couldn't. It's the same reason a random forest beats a single decision tree.

## D2. Sentiment: averaging 5 networks

**Question.** Does averaging help sentiment, where networks barely overfit and a linear model scores the same as a network?

**Experiment.** 5 baseline `[256, 64]` networks, seeds 0–4.

In [18]:
sent_models, sent_singles, sent_curve = seed_ensemble(data['sentiment'], BASE_CFG, seeds=range(5))
print(sent_singles.to_string(index=False))
print(f'\nSingle networks: mean {sent_singles["val_acc"].mean():.4f}\n')
print(sent_curve.to_string(index=False))

training: seed 0
training: seed 1
training: seed 2
training: seed 3
training: seed 4
 seed  best_epoch  val_acc
    0           1   0.6775
    1           2   0.6721
    2           2   0.6654
    3           1   0.6692
    4           2   0.6708

Single networks: mean 0.6710

 n_networks  val_acc
          1   0.6775
          2   0.6758
          3   0.6729
          4   0.6746
          5   0.6713


**Result.** 67.1% vs 67.0% for the single baseline network: no gain.

**Takeaway.** The same technique that gave image about +8.4 points does nothing for sentiment. Averaging only helps when the networks are unstable, and sentiment's aren't. This is further evidence that the BERT features, not the networks, are the limit for sentiment.

# Part E: Closing the gap on image

Our probe submission scored **0.551** on the public leaderboard (validation: 0.566), so the validation estimate held up. The top public leaderboard score was **0.65**. If that team's sentiment is near 67% like ours, its image accuracy is around 62%, against our 46%. The gap is almost entirely on image.

## E1. Looking at the features

**Question.** What do the image features actually look like? So far we had only rescaled them.

In [19]:
X, _, _, _ = load_task('image')
print(f'min {X.min():.3f} | max {X.max():.3f} | mean {X.mean():.3f}')
print(f'exact zeros: {(X == 0).mean():.1%} | negative values: {(X < 0).mean():.1%}')
print(f'non-zero features per image: {(X > 0).sum(axis=1).mean():.0f} of {X.shape[1]}')
print(f'row sums: min {X.sum(axis=1).min():.2f}, max {X.sum(axis=1).max():.2f}')

min 0.000 | max 0.923 | mean 0.015
exact zeros: 85.2% | negative values: 0.0%
non-zero features per image: 74 of 500
row sums: min 2.00, max 12.38


**Result.** Values from 0 to 0.92 (mean 0.015), no negative values, 85.2% exactly zero, row sums from 2.0 to 12.4.

**Takeaway.** The features are sparse, non-negative and histogram-like. Standardizing data like this inflates rare features: a feature that is almost always 0 has a tiny standard deviation, so its occasional non-zero values get blown up.

## E2. Square-root scaling

**Question.** Taking the square root compresses large values and lifts small non-zero ones (0.01 → 0.1, but 0.81 → 0.9). It's the standard preprocessing for histogram-like features. Does it help?

**Experiment.** The current best single network (`[512, 256]`, label smoothing 0.1) with `l2` (the current setup) and the three square-root variants.

In [20]:
sqrt_results = compare_scalings('image', IMAGE_CFG, ['l2'] + SQRT_SCALINGS)
print(sqrt_results.to_string(index=False))

training: image | scaling = l2
training: image | scaling = sqrt
training: image | scaling = sqrt+l2
training: image | scaling = sqrt+standardize
         scaling  best_epoch  train_acc  val_acc    gap
              l2          17        1.0   0.3725 0.6275
            sqrt          21        1.0   0.3495 0.6505
         sqrt+l2          22        1.0   0.3513 0.6487
sqrt+standardize          12        1.0   0.3623 0.6378


**Result.** `l2` 37.3%, `sqrt` 35.0%, `sqrt+l2` 35.1%, `sqrt+standardize` 36.2%.

**Takeaway.** No help, and slightly worse. Every scaling tried so far lands between 35% and 37%, so preprocessing isn't where the gap is. L2 stays.

## E3–E5: Making memorization harder

Every image network still reaches 100% training accuracy, and averaging, which counters that instability, was the one big win. E3–E5 test three other ways of making memorization harder, each on the current best single network (37.3%).

**Adoption rule:** a technique is kept only if it beats the same network without it by more than 1.5 points, the noise level.

## E3. Input dropout

**Question.** Hidden-layer dropout didn't help (B3). Does randomly blanking *input* features help instead? For histogram-like features, that's like hiding some of the visual patterns in each image, so the network can't rely on any single one.

**Experiment.** Input dropout 0, 0.1, 0.2, 0.3, 0.5.

In [21]:
input_drop_results = sweep('input_dropout', [0.0, 0.1, 0.2, 0.3, 0.5], tasks=('image',), base=IMAGE_CFG)
print(input_drop_results.to_string(index=False))

INPUT_DROPOUT, adopted = pick(input_drop_results, 'input_dropout', off_value=0.0)
print(f'\nInput dropout: {INPUT_DROPOUT} ({"adopted" if adopted else "not adopted"})')

training: image | input_dropout = 0.0
training: image | input_dropout = 0.1
training: image | input_dropout = 0.2
training: image | input_dropout = 0.3
training: image | input_dropout = 0.5
 task  input_dropout  best_epoch  train_acc  val_acc    gap
image            0.0          17     1.0000   0.3725 0.6275
image            0.1          36     1.0000   0.3870 0.6130
image            0.2          36     1.0000   0.4010 0.5990
image            0.3          14     0.9928   0.3930 0.5998
image            0.5          59     0.9979   0.4032 0.5946

Input dropout: 0.5 (adopted)


**Result.**

| Input dropout | 0 | 0.1 | 0.2 | 0.3 | 0.5 |
|---|---|---|---|---|---|
| Image val | 37.3% | 38.7% | 40.1% | 39.3% | **40.3%** |

Adopted: 0.5 (+3.1 points). Every non-zero value scored higher than none, and 0.2–0.5 all landed within about a point of each other.

**Takeaway.** This is the first regularizer that improves a single network. Hidden-layer dropout (B3) only reduced memorization, but blanking input features raises accuracy on new data. With sparse, histogram-like features, hiding some of each image's visual patterns forces the network to use many of them instead of a few.

## E4. Mixup

**Question.** Does training on blends of two images help? Each training example is mixed with a random partner, for example 70% of one and 30% of the other, and the label is mixed the same way. The network never sees exactly the same input twice, so it can't memorize individual examples.

**Experiment.** Mixup alpha 0 (off), 0.2, 0.4, 1.0. Higher alpha makes blends closer to 50/50 more often.

In [22]:
mixup_results = sweep('mixup_alpha', [0.0, 0.2, 0.4, 1.0], tasks=('image',), base=IMAGE_CFG)
print(mixup_results.to_string(index=False))

MIXUP_ALPHA, adopted = pick(mixup_results, 'mixup_alpha', off_value=0.0)
print(f'\nMixup alpha: {MIXUP_ALPHA} ({"adopted" if adopted else "not adopted"})')

training: image | mixup_alpha = 0.0
training: image | mixup_alpha = 0.2
training: image | mixup_alpha = 0.4
training: image | mixup_alpha = 1.0
 task  mixup_alpha  best_epoch  train_acc  val_acc    gap
image          0.0          17     1.0000   0.3725 0.6275
image          0.2          14     1.0000   0.3713 0.6287
image          0.4          12     0.9998   0.3698 0.6300
image          1.0          12     0.9958   0.3623 0.6336

Mixup alpha: 0.0 (not adopted)


**Result.** 37.3% without mixup, then 37.1%, 37.0% and 36.2% for alpha 0.2, 0.4 and 1.0. Not adopted.

**Takeaway.** Mixup doesn't help here, and the strongest blending hurts slightly. Blending two sparse feature vectors creates inputs unlike any real image, which may be why it fails where input dropout works.

## E5. Weight decay with longer training

**Question.** Weight decay has been off so far. Does keeping the weights small (AdamW), combined with longer training and a learning rate that gradually falls to 0 (cosine schedule), help?

**Experiment.** AdamW with a cosine schedule over 60 epochs and no early stopping (the best epoch is still kept), weight decay 0, 0.01, 0.05, 0.1. The 0 row shows the effect of longer training with the schedule alone. Every row is compared with the current setup (37.3%).

In [23]:
LONG = dict(optimizer='adamw', cosine=True, epochs=60, patience=60)
wd_results = sweep('weight_decay', [0.0, 0.01, 0.05, 0.1], tasks=('image',), base={**IMAGE_CFG, **LONG})
print(wd_results.to_string(index=False))

reference = lookup(ls_results, label_smoothing=LABEL_SMOOTHING)    # current setup, from C2
WEIGHT_DECAY, adopted = pick(wd_results, 'weight_decay', off_value=None, reference=reference)
LONG_SETTINGS = {**LONG, 'weight_decay': float(WEIGHT_DECAY)} if adopted else {}
print(f'\nReference (current setup): {reference:.4f}')
print(f'Longer training + weight decay: {"adopted, weight decay " + str(WEIGHT_DECAY) if adopted else "not adopted"}')

training: image | weight_decay = 0.0
training: image | weight_decay = 0.01
training: image | weight_decay = 0.05
training: image | weight_decay = 0.1
 task  weight_decay  best_epoch  train_acc  val_acc    gap
image          0.00          25        1.0   0.3740 0.6260
image          0.01          19        1.0   0.3810 0.6190
image          0.05          16        1.0   0.3822 0.6178
image          0.10          30        1.0   0.3822 0.6178

Reference (current setup): 0.3725
Longer training + weight decay: not adopted


**Result.**

| Weight decay | 0 | 0.01 | 0.05 | 0.1 |
|---|---|---|---|---|
| Image val | 37.4% | 38.1% | 38.2% | 38.2% |

Against the current setup's 37.3%, the best gain is +1.0 point, below the 1.5-point threshold. Not adopted.

**Takeaway.** Longer training with the falling learning rate does nothing on its own (the 0 row). Weight decay adds about a point at every value. That's a consistent but unproven gain from single runs, so it's rechecked with 10 networks in E7.

## E6. Combining what worked, then averaging 10 networks

**Question.** Do the adopted techniques stack, and does averaging 10 such networks beat 45.8% (D1)?

**Experiment.** The current setup plus everything adopted in E3–E5, averaged over seeds 0–9 as in D1. If nothing was adopted, the D1 networks are reused.

In [ ]:
FINAL_IMAGE_CFG = {**IMAGE_CFG, 'input_dropout': INPUT_DROPOUT, 'mixup_alpha': MIXUP_ALPHA, **LONG_SETTINGS}
changes = {k: v for k, v in FINAL_IMAGE_CFG.items() if IMAGE_CFG[k] != v}
print('Changes from the D1 setup:', changes if changes else 'none')

if changes:
    final_models, final_singles, final_curve = seed_ensemble(data['image'], FINAL_IMAGE_CFG, seeds=range(10))
else:
    final_models, final_singles, final_curve = image_models, image_singles, image_curve

print(f'\nSingle networks: mean {final_singles["val_acc"].mean():.4f}\n')
print(final_curve.to_string(index=False))
print(f'\nAverage of 10: {lookup(final_curve, n_networks=10):.4f} (D1: {lookup(image_curve, n_networks=10):.4f})')

Changes from the D1 setup: {'input_dropout': 0.5}
training: seed 0
training: seed 1
training: seed 2
training: seed 3
training: seed 4
training: seed 5
training: seed 6
training: seed 7
training: seed 8
training: seed 9

Single networks: mean 0.4015

 n_networks  val_acc
          1   0.4055
          2   0.4515
          3   0.4738
          4   0.4858
          5   0.4908
          6   0.4973
          7   0.4995
          8   0.5032
          9   0.5032
         10   0.5055

Average of 10: 0.5055 (D1: 0.4582)


**Result.** Only input dropout 0.5 was adopted.

| | D1 setup | + input dropout 0.5 |
|---|---|---|
| Single network (mean of 10) | 37.4% | 40.2% |
| Average of 5 | 44.2% | 49.1% |
| Average of 10 | 45.8% | **50.6%** |

**Takeaway.** Input dropout helps the average even more than it helps single networks: +2.8 points for one network, +4.7 for the average of 10. Averaging now adds about 10 points on top of a single network, against about 8.4 before. Input dropout adds randomness during training, so the 10 networks differ more from each other and their mistakes cancel out better.

## E7. Checking near-misses with 10 networks

**Question.** A single run can't tell a real gain of about 1 point from luck. Weight decay with longer training fell just short of the threshold in E5. Does it help when tested the way the model is actually used, as an average of 10 networks?

**Experiment.** The E6 setup plus AdamW, weight decay 0.01, a cosine schedule and 60 epochs. Train 10 networks with seeds 0–9 and compare with E6. Both sides are scored on the same 4,000 validation images, and averaging cancels most of the per-seed luck, so a difference of about 1 point is meaningful here.

In [ ]:
def ensemble_check(n=10, **extra):
    """Train n networks with FINAL_IMAGE_CFG plus `extra` changes and compare their average with E6."""
    cfg = {**FINAL_IMAGE_CFG, **extra}
    models, singles, curve = seed_ensemble(data['image'], cfg, seeds=range(n))
    with_it, without = lookup(curve, n_networks=n), lookup(final_curve, n_networks=n)
    print(f'{extra}: single mean {singles["val_acc"].mean():.4f} | '
          f'average of {n}: {with_it:.4f} vs {without:.4f} without ({with_it - without:+.4f})')
    return cfg, models, singles, curve

E7_CHANGE = dict(optimizer='adamw', cosine=True, epochs=60, patience=60, weight_decay=0.01)
BEST_IMAGE_CFG, best_models, best_singles, best_curve = ensemble_check(**E7_CHANGE)
print()
print(best_curve.to_string(index=False))

**Result.**

| | E6 | + weight decay 0.01, longer training |
|---|---|---|
| Single network (mean of 10) | 40.2% | 42.7% |
| Average of 10 | 50.6% | **51.6%** |

Adopted. This is the final image setup: `[512, 256]`, L2 scaling, label smoothing 0.1, input dropout 0.5, AdamW with weight decay 0.01, a cosine schedule over 60 epochs, averaged over 10 networks.

**Takeaway.** The gain that looked like noise in E5 is real: +2.6 points per network, measured over 10 seeds, and +1.0 for the average. A single run can hide a small but real effect, and checking at the level the model is actually used settles it. Because this check was chosen after seeing E5, 51.6% is slightly optimistic, like every validation number here.

*The code cell above was not run in the saved copy of this notebook. The same final image setup is trained again in `comparison.ipynb` (Section 8), which reproduces 51.6% for the average of 10 networks.*


# Conclusions

*The summary-table code cell below was not run in the saved copy; the text after it gives the numbers from the runs above.*


In [ ]:
image_steps = [
    ('Untrained network',                              untrained['image']),
    ('Baseline [256, 64]',                             baseline['image']['best']['val_acc']),
    ('Wider [512, 256]',                               lookup(arch_results, task='image', hidden='[512, 256]')),
    (f'+ {SCALING} scaling',                           lookup(scaling_results, scaling=SCALING)),
    (f'+ label smoothing {LABEL_SMOOTHING}',           lookup(ls_results, label_smoothing=LABEL_SMOOTHING)),
    ('+ average of 10 networks (D1)',                  lookup(image_curve, n_networks=10)),
    ('+ input dropout 0.5, single (mean, E6)',         float(final_singles['val_acc'].mean())),
    ('+ input dropout 0.5, average of 10 (E6)',        lookup(final_curve, n_networks=10)),
    ('+ weight decay, longer training, single (E7)',   float(best_singles['val_acc'].mean())),
    ('+ weight decay, longer training, avg of 10 (E7)', lookup(best_curve, n_networks=10)),
]
sentiment_steps = [
    ('Untrained network',                              untrained['sentiment']),
    ('Baseline [256, 64]',                             baseline['sentiment']['best']['val_acc']),
    ('Linear (no hidden layers)',                      lookup(arch_results, task='sentiment', hidden='[]')),
    ('Average of 5 baseline networks',                 lookup(sent_curve, n_networks=5)),
]
for name, steps in [('IMAGE', image_steps), ('SENTIMENT', sentiment_steps)]:
    print(name)
    for label, acc in steps:
        print(f'  {label:50s} {acc:.4f}')
    print()

best_image = max(acc for _, acc in image_steps)
best_sent = max(acc for _, acc in sentiment_steps)
print(f'Best overall validation score (image + sentiment) / 2 = {(best_image + best_sent) / 2:.4f}')

**Image: 33.8% → 51.6%**
- Training takes the network from chance (2%) to 33.8%, but it memorizes the training set.
- Hidden-layer dropout, the learning rate, feature scaling (including square root), label smoothing and mixup each change the result by less than the noise, or make it worse.
- What worked, in order of impact:
  1. **Averaging networks** (D1): 37.4% → 45.8% with 10. It counters the instability that comes with memorizing.
  2. **Input dropout 0.5** (E3, E6): the only regularizer that helped a single network (+3 points), and it made averaging even stronger, reaching 50.6%.
  3. **A wider network** (B5): +2.6 points.
  4. **Weight decay with longer training** (E7): +1.0 point, invisible in single runs and confirmed with 10 networks, reaching 51.6%.
- The features are sparse and histogram-like (85% zeros). Techniques that work *with* that structure, like hiding some features, helped. Techniques that reshape the values (square root) or blend examples (mixup) didn't.

**Sentiment: 67%, set by the features**
- About 67% regardless of dropout, learning rate, network size or averaging, and a linear model matches the network.
- The limit appears to be how well the frozen BERT features separate the three sentiments, not the classifier.

**Overall (this notebook):** the validation score rose from 0.504 (baseline) to **0.594**: the average of 10 networks with the final setup for image (51.6%), and the linear model for sentiment (67.3%), the simplest of three sentiment options tied within noise.

**Final submission:** the final model was chosen in `comparison.ipynb` (Section 13), among the four required algorithms on the same split: k-NN for image (56.3%) and this notebook's average of 5 `[256, 64]` networks for sentiment (67.1%, tied with the linear model). Expected 0.617 on validation; it scored 0.628 on the public leaderboard.

**Validation vs leaderboard:** the earlier 0.566 setup scored 0.551 on the public leaderboard, a small drop consistent with leaderboard noise and with the slight optimism of making every choice on the validation set. The final submission went the other way (0.617 on validation, 0.628 on the leaderboard), so both gaps are within leaderboard noise.